# Business Forecasting · Time-Series Regression in Python
**New, self-contained student practice notebook** · Amir Hosseinipour

This is an original companion exercise prepared for the course website. It is **not** a copy of the previously created Lecture 5 notebook, which was unavailable for direct transfer. All sales data in this notebook are **simulated for teaching**; do not present them as real company observations.

**Learning goals:** explain trend and seasonal dummies; estimate a regression; use a chronological validation set; interpret coefficients and consider a manager's forecast scenario.

## 1. Set up and generate a realistic example
We simulate 48 months of sales for a retailer. Explanatory variables include price, advertising expenditure, time trend, and quarter. The results are repeatable because the random generator has a fixed seed. In a real project, you would replace this section with a verified dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

rng = np.random.default_rng(477)
months = pd.date_range('2023-01-01', periods=48, freq='MS')
t = np.arange(1, 49)
quarter = months.quarter.astype(int)
price = 14.0 + 0.3 * np.sin(2*np.pi*t/12) + 0.018*t
ad_spend = 42 + 7*np.sin(2*np.pi*(t+1)/12) + rng.normal(0, 2.5, len(t))
season_effect = np.select([quarter==1, quarter==2, quarter==3, quarter==4], [0,24,35,58])
sales = 210 + 4.5*t + 5.0*ad_spend - 9.0*price + season_effect + rng.normal(0, 12, len(t))
df = pd.DataFrame({'month':months, 't':t, 'quarter':quarter, 'price':price, 'ad_spend':ad_spend, 'sales':sales})
print(df.head().round(2))

**Interpretation:** We constructed an increasing underlying trend and a seasonal pattern. The role of regression is to estimate these effects from the observed data, not to assume that coefficients are known.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df['month'], df['sales'], marker='o', markersize=3)
ax.set(title='Illustrative retail sales', xlabel='Month', ylabel='Units sold')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Split by time, not at random
The first 36 months become the training sample; the final 12 months are the validation period. A random split would give the model access to observations from the future.

In [ ]:
train = df.iloc[:36].copy()
test = df.iloc[36:].copy()
print('Train:', train['month'].min().date(), 'to', train['month'].max().date())
print('Test: ', test['month'].min().date(), 'to', test['month'].max().date())

## 3. Fit an explanatory regression
The expression `C(quarter)` automatically creates three dummy variables when there are four quarters and an intercept. The omitted quarter becomes the reference category.

In [ ]:
model = smf.ols('sales ~ t + price + ad_spend + C(quarter)', data=train).fit()
print(model.summary())

**Questions:** Which coefficient represents a one-unit price increase, holding other predictors constant? Which quarter is the reference category? Does a positive association prove advertising causes higher sales? (No: observational confounding remains possible.)

## 4. Forecast the validation period
For an *ex ante* real forecast, the future predictor values must be known or forecast separately. Here we use the simulated future price and advertising values to isolate the model demonstration; this is a **scenario-style evaluation**, not a claim of fully operational ex ante accuracy.

In [ ]:
pred_test = model.predict(test)
rmse = float(np.sqrt(np.mean((test['sales'].to_numpy() - pred_test.to_numpy())**2)))
mae = float(np.mean(np.abs(test['sales'].to_numpy() - pred_test.to_numpy())))
print(f'Validation RMSE: {rmse:.2f} units')
print(f'Validation MAE: {mae:.2f} units')
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(train['month'], train['sales'], label='Training', color='steelblue')
ax.plot(test['month'], test['sales'], label='Actual holdout', color='black', marker='o')
ax.plot(test['month'], pred_test, label='Regression prediction', color='darkred', marker='x')
ax.legend(); ax.grid(alpha=0.2); ax.set_ylabel('Units sold'); plt.tight_layout(); plt.show()

## 5. Residual checks
Inspect the residual pattern. A systematic remaining trend or seasonality indicates the model is missing structure. A high R² alone does not establish forecast quality.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(train['month'], model.resid, marker='o', markersize=3)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set(title='Training residuals over time', xlabel='Month')
axes[1].scatter(model.fittedvalues, model.resid, alpha=.7)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(title='Residuals versus fitted values', xlabel='Fitted sales')
plt.tight_layout(); plt.show()

## 6. A pricing scenario for a manager
Assume marketing expects advertising expenditure of 45 units in each of the next three months. Compare model predictions at prices of 14 and 15. This is *not* an optimized pricing recommendation: margins, competitor reactions, uncertainty, causality, and operational constraints would also matter.

In [ ]:
future_months = pd.date_range(df['month'].max() + pd.offsets.MonthBegin(1), periods=3, freq='MS')
future_base = pd.DataFrame({'month':future_months, 't':np.arange(49,52), 'quarter':future_months.quarter.astype(int), 'ad_spend':45.0})
low = future_base.assign(price=14.0)
high = future_base.assign(price=15.0)
scenario = pd.DataFrame({'month':future_months.strftime('%Y-%m'), 'units_at_14':model.predict(low).round(1), 'units_at_15':model.predict(high).round(1)})
scenario['difference_in_units'] = (scenario['units_at_15'] - scenario['units_at_14']).round(1)
print(scenario.to_string(index=False))

## Exit questions
1. Why should the validation period come **after** the training period?
2. How do you interpret a quarterly dummy coefficient?
3. What information would you need before turning the two pricing scenarios into a real decision?
4. What would change in an *ex ante* forecast when advertising is not yet known?

**Key caution:** The dataset is illustrative. The coefficients are associations fitted to generated observations, not causal effects measured in a real company.